In [0]:
#path configuration
#testing using iclr24v2 and iclr25v2 to verify against known answers

DATA_PATH = "/Volumes/workspace/default/ijc427/iclr26v1.parquet"
#DATA_PATH = "/Volumes/workspace/default/ijc427/iclr24v2.parquet"
#DATA_PATH = "/Volumes/workspace/default/ijc427/iclr25v2.parquet"

OA_PATH = "/Volumes/workspace/default/ijc427/openalex.json"

from pyspark.sql import functions as F
from pyspark.sql.functions import col

iclr_df     = spark.read.parquet(DATA_PATH)
openalex_df = spark.read.option("multiLine", True).json(OA_PATH)

print(f"Loaded: {DATA_PATH}")

#counting rows
print(f"ICLR rows:     {iclr_df.count()}")
print(f"OpenAlex rows: {openalex_df.count()}")

#printing schemas
print("\n--- ICLR schema ---")
iclr_df.printSchema()

print("\n--- OpenAlex schema ---")
openalex_df.printSchema()

#sampling rows from each dataset
print("\n--- ICLR sample ---")
iclr_df.show(3, truncate=False)

print("\n--- OpenAlex sample ---")
openalex_df.show(3, truncate=False)


Loaded: /Volumes/workspace/default/ijc427/iclr26v1.parquet
ICLR rows:     55906
OpenAlex rows: 2245

--- ICLR schema ---
root
 |-- year: long (nullable = true)
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- author_ids: string (nullable = true)
 |-- decision: string (nullable = true)
 |-- scores: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- labels: string (nullable = true)


--- OpenAlex schema ---
root
 |-- apc_list: struct (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- value: long (nullable = true)
 |    |-- value_usd: long (nullable = true)
 |-- apc_paid: struct (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- value: long (nullable = true)
 |    |-- value_usd: long (nullable = true)
 |-- authorships: array (nullab

In [0]:
#inspecting ICLR dataset including columns needed to answer the questions

#looking at scores
print("--- SCORES ---")
#seeing score values
iclr_df.select("id", "scores").show(5, truncate=False)

#confirming if the scores are arrays
iclr_df.select(
    F.array_min(F.col("scores")).alias("min_score"),
    F.array_max(F.col("scores")).alias("max_score"),
    F.size(F.col("scores")).alias("num_scores")
).show(5)

#seeing score range across all papers
iclr_df.withColumn("score", F.explode(F.col("scores"))) \
    .select("score") \
    .summary("min", "max", "mean", "50%").show()

#seeing how many reviewers there are per paper
iclr_df.select(F.size(F.col("scores")).alias("num_reviewers")) \
    .summary("min", "max", "mean").show()

#seeing keywords
print("--- KEYWORDS ---")
#seeing the keywords values
iclr_df.select("id", "keywords").show(5, truncate=False)

#seeing how many keywords there are per paper
iclr_df.select(F.size(F.col("keywords")).alias("num_keywords")) \
    .summary("min", "max", "mean").show()

#viewing sample of individual keywords after exploding
iclr_df.withColumn("keyword", F.explode(F.col("keywords"))) \
    .select("keyword").distinct() \
    .orderBy("keyword").show(20, truncate=False)

#seeing titles
print("--- TITLES ---")
iclr_df.select("id", "title").show(5, truncate=False)

#seeing years
print("--- YEARS ---")
iclr_df.select("year").distinct().orderBy("year").show()

# checking for nulls in key columns needed
print("--- NULL CHECKS ---")
print("Null scores:   ", iclr_df.filter(F.col("scores").isNull()).count())
print("Null keywords: ", iclr_df.filter(F.col("keywords").isNull()).count())
print("Null title:    ", iclr_df.filter(F.col("title").isNull()).count())
print("Null year:     ", iclr_df.filter(F.col("year").isNull()).count())
print("Null decision: ", iclr_df.filter(F.col("decision").isNull()).count())

--- SCORES ---
+---------+---------+
|id       |scores   |
+---------+---------+
|B1-Hhnslg|[6, 4, 5]|
|B1-q5Pqxl|[6, 6, 7]|
|B16Jem9xe|[8, 7, 6]|
|B16dGcqlx|[6, 5, 6]|
|B184E5qee|[7, 9, 5]|
+---------+---------+
only showing top 5 rows
+---------+---------+----------+
|min_score|max_score|num_scores|
+---------+---------+----------+
|        4|        6|         3|
|        6|        7|         3|
|        6|        8|         3|
|        5|        6|         3|
|        5|        9|         3|
+---------+---------+----------+
only showing top 5 rows
+-------+-----------------+
|summary|            score|
+-------+-----------------+
|    min|                1|
|    max|               10|
|   mean|5.135199240986717|
|    50%|                5|
+-------+-----------------+

+-------+------------------+
|summary|     num_reviewers|
+-------+------------------+
|    min|                 0|
|    max|                12|
|   mean|2.4131935749293456|
+-------+------------------+

--- KEYWORDS 

In [0]:
#inspecting openAlex dataset
#checking 2 fields needed to answer Q5 and Q6

#seeing if the title is at the top level
openalex_df.select("title").show(5, truncate=False)

#checking if the citation field is named 'cited_by_count'
openalex_df.select("cited_by_count").show(5)

#checking for null values
print("Null titles in OpenAlex:",
    openalex_df.filter(F.col("title").isNull()).count())
print("Null cited_by_count:",
    openalex_df.filter(F.col("cited_by_count").isNull()).count())

#viewing the publication years
openalex_df.select("publication_year").distinct() \
    .orderBy("publication_year").show()

#looking at citation distribution
openalex_df.select("cited_by_count") \
    .summary("min", "max", "mean", "50%").show()

+-------------------------------------------------------------------------------------+
|title                                                                                |
+-------------------------------------------------------------------------------------+
|beta-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework    |
|Optimization as a Model for Few-Shot Learning                                        |
|GLUE: A Multi-Task Benchmark and Analysis Platform for Natural Language Understanding|
|A Simple but Tough-to-Beat Baseline for Sentence Embeddings                          |
|DEBERTA: DECODING-ENHANCED BERT WITH DISENTANGLED ATTENTION                          |
+-------------------------------------------------------------------------------------+
only showing top 5 rows
+--------------+
|cited_by_count|
+--------------+
|          3085|
|          2440|
|          1864|
|          1051|
|           922|
+--------------+
only showing top 5 rows
Null ti

In [0]:
#viewing decision values
#looking at all unique values before mapping for Q1

print("All unique decision values:")
iclr_df.select("decision") \
    .distinct() \
    .orderBy("decision") \
    .show(100, truncate=False)

print(f"\nNull decisions: {iclr_df.filter(F.col('decision').isNull()).count()}")

print("\nRaw decision counts:")
iclr_df.groupBy("decision").count() \
    .orderBy("count", ascending=False) \
    .show(50, truncate=False)

All unique decision values:
+------------------------+
|decision                |
+------------------------+
|                        |
|Accept (Oral)           |
|Accept (Poster)         |
|Accept (Spotlight)      |
|Accept (Talk)           |
|Accept (oral)           |
|Accept (poster)         |
|Accept (spotlight)      |
|Accept: notable-top-25% |
|Accept: notable-top-5%  |
|Accept: poster          |
|Desk rejected           |
|Invite to Workshop Track|
|Reject                  |
|Withdrawn               |
+------------------------+


Null decisions: 0

Raw decision counts:
+------------------------+-----+
|decision                |count|
+------------------------+-----+
|                        |19673|
|Reject                  |17096|
|Withdrawn               |7594 |
|Accept (Poster)         |6173 |
|Accept (poster)         |1809 |
|Accept: poster          |1202 |
|Accept (Spotlight)      |775  |
|Accept (Oral)           |383  |
|Accept (spotlight)      |366  |
|Accept: notable-top-

In [0]:
#checking interoperability
#seeing if ICLR titles match openAlex titles

#normalising the titles in both datasets to lowercase to make matching easier
iclr_titles = iclr_df.withColumn(
    "title_norm", F.lower(F.trim(F.col("title")))
).select("title_norm")

oa_titles = openalex_df.withColumn(
    "title_norm", F.lower(F.trim(F.col("title")))
).select("title_norm")

#viewing how many titles are in each dataset
print(f"ICLR titles:     {iclr_titles.count()}")
print(f"OpenAlex titles: {oa_titles.count()}")

#viewing how many of these titles match between datasets
matched = iclr_titles.intersect(oa_titles)
print(f"Matched titles:  {matched.count()}")

#seeing what percentage of openAlex dataset titles can be matched to ICLR
oa_count = oa_titles.count()
match_count = matched.count()
print(f"Match rate: {round(match_count/oa_count*100, 1)}% of OpenAlex papers matched to ICLR")


ICLR titles:     55906
OpenAlex titles: 2245
Matched titles:  1591
Match rate: 70.9% of OpenAlex papers matched to ICLR


In [0]:
#Q1: counting accepted, rejected and withdrawn papers
#decision column contains 15 unique string variants
#"Invite to Workshop Track" = acceptance
#19,673 papers have a blank decision (no decision yet at scrape time). This is classed as Pending and excluded from Q1 counts.

iclr_cleaned = iclr_df.withColumn("standard_decision",
    F.when(
        F.lower(F.col("decision")).contains("accept") |
        F.lower(F.col("decision")).contains("invite"),
        "Accepted"
    ).when(
        F.lower(F.col("decision")).contains("reject"),
        "Rejected"
    ).when(
        F.lower(F.col("decision")).contains("withdrawn"),
        "Withdrawn"
    ).when(
        F.col("decision") == "",
        "Pending"
    ).otherwise("Other")
)

#verifying the mapping, checking that 'other' = 0
print("--- Mapping verification (Other must be 0) ---")
iclr_cleaned.groupBy("standard_decision").count() \
    .orderBy("count", ascending=False).show()

#answering the question (not counting Pending)
print("--- Q1 RESULT: Accepted / Rejected / Withdrawn ---")
iclr_cleaned.filter(F.col("standard_decision") != "Pending") \
    .groupBy("standard_decision").count() \
    .orderBy("standard_decision").show()


--- Mapping verification (Other must be 0) ---
+-----------------+-----+
|standard_decision|count|
+-----------------+-----+
|          Pending|19673|
|         Rejected|17290|
|         Accepted|11349|
|        Withdrawn| 7594|
+-----------------+-----+

--- Q1 RESULT: Accepted / Rejected / Withdrawn ---
+-----------------+-----+
|standard_decision|count|
+-----------------+-----+
|         Accepted|11349|
|         Rejected|17290|
|        Withdrawn| 7594|
+-----------------+-----+



In [0]:
#Q1 EXTENSION: Acceptance rate by year
#adding temporal dimension
#this shows whether ICLR is becoming more or less competitive over time

#grouping by year and decision
#counting papers
yearly = iclr_cleaned \
    .filter(F.col("standard_decision") != "Pending") \
    .groupBy("year", "standard_decision") \
    .count()

#pivoting so each decision is its own column
yearly_pivot = yearly.groupBy("year") \
    .pivot("standard_decision", 
           ["Accepted", "Rejected", "Withdrawn"]) \
    .sum("count") \
    .fillna(0) \
    .orderBy("year")

#calculating acceptance rate
#denominator = Accepted + Rejected only (excludes Withdrawn as these papers never received a final accept/reject decision)
yearly_with_rate = yearly_pivot.withColumn(
    "total_decided",
    F.col("Accepted") + F.col("Rejected")
).withColumn(
    "acceptance_rate_%",
    F.round(
        F.when(F.col("total_decided") != 0, F.col("Accepted") / F.col("total_decided") * 100).otherwise(0), 1
    )
)

print("--- EXTENSION: Acceptance rate by year ---")
yearly_with_rate.select(
    "year", "Accepted", "Rejected", "Withdrawn",
    "total_decided", "acceptance_rate_%"
).show()

--- EXTENSION: Acceptance rate by year ---
+----+--------+--------+---------+-------------+-----------------+
|year|Accepted|Rejected|Withdrawn|total_decided|acceptance_rate_%|
+----+--------+--------+---------+-------------+-----------------+
|2017|     245|     244|        0|          489|             50.1|
|2018|     425|     492|       95|          917|             46.3|
|2019|     502|     917|      150|         1419|             35.4|
|2020|     687|    1537|      369|         2224|             30.9|
|2021|     859|    1752|      398|         2611|             32.9|
|2022|    1094|    1551|      777|         2645|             41.4|
|2023|    1573|    2244|     1138|         3817|             41.2|
|2024|    2261|    3539|     1601|         5800|             39.0|
|2025|    3703|    5014|     2946|         8717|             42.5|
|2026|       0|       0|      120|            0|              0.0|
+----+--------+--------+---------+-------------+-----------------+



In [0]:
#Q2: finding accepted papers with at least once score under 5
#scores is array<long>
#array_min() returns null for empty arrays (0-score papers)

accepted_low = iclr_cleaned.filter(
    (F.col("standard_decision") == "Accepted") &
    (F.array_min(F.col("scores")) < 5)
)

#answering question
print(f"Accepted papers with at least one score < 5: {accepted_low.count()}")

Accepted papers with at least one score < 5: 1596


In [0]:
#Q2 EXTENSION: logistic regression
#can reviewer scores predict whether a paper is accepted?

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import stddev, avg, size

#building feature dataset
#features used: mean score, score std dev (reviewer disagreement), number of reviewers
#labels: 1 = Accepted, 0 = Rejected (excluding Withdrawn + Pending)

#computing per-paper stats from scores
scores_stats = iclr_cleaned \
    .filter(F.size(F.col("scores")) > 0) \
    .withColumn("score", F.explode(F.col("scores"))) \
    .withColumn("score", F.col("score").cast("double")) \
    .groupBy("id", "standard_decision") \
    .agg(
        avg("score").alias("mean_score"),
        stddev("score").alias("score_std"),
        F.count("score").alias("num_reviewers")
    ) \
    .fillna({"score_std": 0.0})

#keeping only accepted and rejected papers
ml_input = scores_stats.filter(
    F.col("standard_decision").isin(["Accepted", "Rejected"])
).withColumn(
    "label",
    F.when(F.col("standard_decision") == "Accepted", 1.0)
     .otherwise(0.0)
)

print(f"Training data size: {ml_input.count()} papers")
ml_input.groupBy("label").count().show()

#putting all features into a single vector
assembler = VectorAssembler(
    inputCols=["mean_score", "score_std", "num_reviewers"],
    outputCol="features"
)
ml_data = assembler.transform(ml_input) \
    .select("features", "label") \
    .dropna()

#splitting training and testing sets (80/20)
train, test = ml_data.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count()} rows, Test: {test.count()} rows")

#fitting logistic regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100
)
lr_model = lr.fit(train)

#evaluating model on test set and calculating AUC
predictions = lr_model.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="label")
auc = evaluator.evaluate(predictions)
print(f"\nAUC (Area Under ROC Curve): {auc:.4f}")

#interpreting coefficients
print("\nModel coefficients:")
print(f"  mean_score:    {lr_model.coefficients[0]:.4f}")
print(f"  score_std:     {lr_model.coefficients[1]:.4f}")
print(f"  num_reviewers: {lr_model.coefficients[2]:.4f}")
print(f"  intercept:     {lr_model.intercept:.4f}")

#viewing confusion matrix
print("\nConfusion matrix:")
predictions.groupBy("label", "prediction").count() \
    .orderBy("label", "prediction").show()

Training data size: 28476 papers
+-----+-----+
|label|count|
+-----+-----+
|  1.0|11349|
|  0.0|17127|
+-----+-----+

Train: 22808 rows, Test: 5668 rows

AUC (Area Under ROC Curve): 0.9418

Model coefficients:
  mean_score:    3.2250
  score_std:     -0.0565
  num_reviewers: -0.1527
  intercept:     -18.1701

Confusion matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 3039|
|  0.0|       1.0|  318|
|  1.0|       0.0|  413|
|  1.0|       1.0| 1898|
+-----+----------+-----+



In [0]:
#Q3: calculating top 10 papers by mean reviewer score
from pyspark.sql.functions import explode, avg

#exploding scores into separate rows
scores_flat = iclr_cleaned \
    .filter(F.size(F.col("scores")) > 0) \
    .withColumn("score", explode(F.col("scores")))

#casting in a separate withColumn call
scores_flat = scores_flat \
    .withColumn("score", F.col("score").cast("double"))

#computing mean per paper
mean_scores = scores_flat.groupBy("id", "year") \
    .agg(avg("score").alias("mean_score"))

#answering the question
top10 = mean_scores \
    .orderBy(F.col("mean_score").desc()) \
    .limit(10) \
    .select("id", "mean_score", "year")


top10.show(truncate=False)

+-----------+-----------------+----+
|id         |mean_score       |year|
+-----------+-----------------+----+
|u1cQYxRI1H |10.0             |2025|
|Sy8gdB9xx  |9.666666666666666|2017|
|6Mxhg9PtDE |9.5              |2025|
|88nT0j5jAn |9.333333333333334|2023|
|LyJi5ugyJx |9.2              |2025|
|WCRQFlji2q |9.0              |2025|
|DJSZGGZYVi |9.0              |2025|
|Uuf2q9TfXGA|9.0              |2023|
|YrycTjllL0 |9.0              |2025|
|aN4Jf6Cx69 |9.0              |2024|
+-----------+-----------------+----+



In [0]:
#Q3 EXTENSION: reviewer disagreement analysis
#examining whether reviewer disagreement (score std dev) differs between accepted and rejected papers

from pyspark.sql.functions import stddev, avg, count as spark_count

#computing per-paper score statistics
score_stats = iclr_cleaned \
    .filter(F.size(F.col("scores")) > 0) \
    .withColumn("score", F.explode(F.col("scores"))) \
    .withColumn("score", F.col("score").cast("double")) \
    .groupBy("id", "year", "standard_decision") \
    .agg(
        avg("score").alias("mean_score"),
        stddev("score").alias("score_std"),
        spark_count("score").alias("num_reviewers")
    ).fillna({"score_std": 0.0})

#comparing mean score and std dev by decision category
print("--- Score statistics by decision category ---")
score_stats.filter(
    F.col("standard_decision").isin(["Accepted", "Rejected"])
).groupBy("standard_decision").agg(
    F.round(avg("mean_score"), 3).alias("avg_mean_score"),
    F.round(avg("score_std"), 3).alias("avg_score_std"),
    F.round(avg("num_reviewers"), 2).alias("avg_reviewers"),
    spark_count("id").alias("num_papers")
).orderBy("standard_decision").show()

#seeing what % of papers score in each range
print("--- Score distribution across all papers ---")
score_stats.withColumn("score_band",
    F.when(F.col("mean_score") < 3, "Below 3")
     .when(F.col("mean_score") < 4, "3 to 4")
     .when(F.col("mean_score") < 5, "4 to 5")
     .when(F.col("mean_score") < 6, "5 to 6")
     .when(F.col("mean_score") < 7, "6 to 7")
     .when(F.col("mean_score") < 8, "7 to 8")
     .otherwise("8 and above")
).groupBy("score_band", "standard_decision").count() \
 .filter(F.col("standard_decision").isin(["Accepted", "Rejected"])) \
 .orderBy("score_band", "standard_decision").show()

#examining what mean score predicts acceptance
print("--- Acceptance rate by mean score band ---")
score_stats.filter(
    F.col("standard_decision").isin(["Accepted", "Rejected"])
).withColumn("score_band",
    F.when(F.col("mean_score") < 4, "Below 4")
     .when(F.col("mean_score") < 5, "4 to 5")
     .when(F.col("mean_score") < 6, "5 to 6")
     .when(F.col("mean_score") < 7, "6 to 7")
     .otherwise("7 and above")
).groupBy("score_band").agg(
    spark_count(F.when(F.col("standard_decision") == "Accepted",
        True)).alias("accepted"),
    spark_count(F.when(F.col("standard_decision") == "Rejected",
        True)).alias("rejected")
).withColumn("acceptance_rate_%",
    F.round(F.col("accepted") /
    (F.col("accepted") + F.col("rejected")) * 100, 1)
).orderBy("score_band").show()

--- Score statistics by decision category ---
+-----------------+--------------+-------------+-------------+----------+
|standard_decision|avg_mean_score|avg_score_std|avg_reviewers|num_papers|
+-----------------+--------------+-------------+-------------+----------+
|         Accepted|         6.494|        1.086|         3.77|     11349|
|         Rejected|         4.676|        1.222|         3.77|     17127|
+-----------------+--------------+-------------+-------------+----------+

--- Score distribution across all papers ---
+-----------+-----------------+-----+
| score_band|standard_decision|count|
+-----------+-----------------+-----+
|     3 to 4|         Accepted|    5|
|     3 to 4|         Rejected| 2520|
|     4 to 5|         Accepted|  103|
|     4 to 5|         Rejected| 5548|
|     5 to 6|         Accepted| 1824|
|     5 to 6|         Rejected| 6742|
|     6 to 7|         Accepted| 6220|
|     6 to 7|         Rejected| 1489|
|     7 to 8|         Accepted| 2643|
|     7 

In [0]:
#Q4: seeing which keywords are growing from <5 to >50 the year following
#keywords are array<string>
#normalising keywords to lowercase and trimming whitespace for easier counting
#filtering out papers with 0 keywords before exploding

from pyspark.sql.functions import explode

#exploding keywords into individual rows
keywords_df = iclr_cleaned \
    .filter(F.size(F.col("keywords")) > 0) \
    .withColumn("keyword", explode(F.col("keywords"))) \
    .withColumn("keyword", F.lower(F.trim(F.col("keyword")))) \
    .select("year", "keyword")

#counting per keyword per year
keyword_counts = keywords_df.groupBy("year", "keyword").count()

#self-join on same keyword with consecutive years
kc1 = keyword_counts.alias("y1")
kc2 = keyword_counts.alias("y2")

growth = kc1.join(kc2,
    (F.col("y1.keyword") == F.col("y2.keyword")) &
    (F.col("y1.year") + 1 == F.col("y2.year")),
    "inner"
).filter(
    (F.col("y1.count") < 5) &
    (F.col("y2.count") > 50)
).select(
    F.col("y1.keyword").alias("keyword"),
    F.col("y1.year").alias("year_low"),
    F.col("y1.count").alias("count_low"),
    F.col("y2.year").alias("year_high"),
    F.col("y2.count").alias("count_high")
).orderBy(F.col("count_high").desc()).limit(10)

growth.show(truncate=False)

+----------------------+--------+---------+---------+----------+
|keyword               |year_low|count_low|year_high|count_high|
+----------------------+--------+---------+---------+----------+
|test-time scaling     |2025    |3        |2026     |87        |
|llm                   |2023    |4        |2024     |76        |
|large reasoning models|2025    |1        |2026     |57        |
|spatial reasoning     |2025    |3        |2026     |57        |
+----------------------+--------+---------+---------+----------+



In [0]:
#Q4 EXTENSION: k-means clustering of papers by keyword
#grouping papers into research communities based on keyword profiles
#silhouette score used to find optimal k

from pyspark.ml.feature import HashingTF, IDF, Tokenizer
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

#preparing keyword string per paper
#joining array into a single space-separated string for tokenizer
keyword_docs = iclr_cleaned \
    .filter(F.size(F.col("keywords")) > 0) \
    .withColumn(
        "keyword_string",
        F.concat_ws(" ", F.col("keywords"))
    ).select("id", "year", "standard_decision", "keyword_string")

print(f"Papers with keywords: {keyword_docs.count()}")

#splitting keyword string back into words
tokenizer = Tokenizer(
    inputCol="keyword_string",
    outputCol="words"
)
tokenized = tokenizer.transform(keyword_docs)

#looking at how often each word appears
hashing_tf = HashingTF(
    inputCol="words",
    outputCol="raw_features",
    numFeatures=200
)
featurised = hashing_tf.transform(tokenized)

#downweighting words that appear in every paper
idf = IDF(inputCol="raw_features", outputCol="features")
idf_model = idf.fit(featurised)
vectorised = idf_model.transform(featurised)

print("TF-IDF vectorisation complete")

#finding optimal k using silhouette score
print("\n--- Silhouette scores by k ---")
evaluator = ClusteringEvaluator(
    featuresCol="features",
    metricName="silhouette"
)

for k in [3, 4, 5, 6, 7]:
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    model = kmeans.fit(vectorised)
    predictions = model.transform(vectorised)
    score = evaluator.evaluate(predictions)
    print(f"k={k}  silhouette={score:.4f}")

Papers with keywords: 53775
TF-IDF vectorisation complete

--- Silhouette scores by k ---
k=3  silhouette=0.0581
k=4  silhouette=0.0544
k=5  silhouette=-0.0101
k=6  silhouette=-0.0067
k=7  silhouette=-0.0096


In [0]:
#Q4 EXTENSION: examining the clusters
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

#fitting final model with k=3
kmeans_final = KMeans(featuresCol="features", k=3, seed=42)
final_model = kmeans_final.fit(vectorised)
clustered = final_model.transform(vectorised)

#seeing cluster sizes
print("--- Cluster sizes ---")
clustered.groupBy("prediction").count() \
    .orderBy("prediction").show()

#finding acceptance rate per cluster
print("--- Acceptance rate per cluster ---")
cluster_decisions = clustered \
    .filter(F.col("standard_decision") != "Pending") \
    .groupBy("prediction", "standard_decision") \
    .count()

cluster_pivot = cluster_decisions.groupBy("prediction") \
    .pivot("standard_decision",
           ["Accepted", "Rejected", "Withdrawn"]) \
    .sum("count") \
    .fillna(0) \
    .orderBy("prediction")

cluster_pivot.withColumn(
    "acceptance_rate_%",
    F.round(
        F.when(
            (F.col("Accepted") + F.col("Rejected")) > 0,
            F.col("Accepted") /
            (F.col("Accepted") + F.col("Rejected")) * 100
        ).otherwise(0), 1
    )
).show()

#finding top keywords per cluster
#joining clustered back to iclr_cleaned to get keywords column
print("--- Top 10 keywords per cluster ---")
clustered_with_kw = clustered.select("id", "prediction") \
    .join(
        iclr_cleaned.select("id", "keywords"),
        on="id",
        how="inner"
    )

cluster_keywords = clustered_with_kw \
    .filter(F.size(F.col("keywords")) > 0) \
    .withColumn("keyword", F.explode(F.col("keywords"))) \
    .withColumn("keyword", F.lower(F.trim(F.col("keyword")))) \
    .groupBy("prediction", "keyword") \
    .count()

window = Window.partitionBy("prediction") \
    .orderBy(F.col("count").desc())

cluster_keywords \
    .withColumn("rank", rank().over(window)) \
    .filter(F.col("rank") <= 10) \
    .orderBy("prediction", "rank") \
    .show(30, truncate=False)

#finding year distribution per cluster
print("--- Year distribution per cluster ---")
clustered.groupBy("prediction", "year") \
    .count() \
    .orderBy("prediction", "year") \
    .show(40)

--- Cluster sizes ---
+----------+-----+
|prediction|count|
+----------+-----+
|         0|47102|
|         1| 3525|
|         2| 3148|
+----------+-----+

--- Acceptance rate per cluster ---
+----------+--------+--------+---------+-----------------+
|prediction|Accepted|Rejected|Withdrawn|acceptance_rate_%|
+----------+--------+--------+---------+-----------------+
|         0|    9577|   14306|     6361|             40.1|
|         1|     797|    1326|      520|             37.5|
|         2|     429|     542|      244|             44.2|
+----------+--------+--------+---------+-----------------+

--- Top 10 keywords per cluster ---
+----------+-------------------------------+-----+----+
|prediction|keyword                        |count|rank|
+----------+-------------------------------+-----+----+
|0         |reinforcement learning         |2818 |1   |
|0         |deep learning                  |2299 |2   |
|0         |large language models          |2284 |3   |
|0         |representa

In [0]:
#Q5: basic join with no normalisation and no deduplication
#matching approach used in provided solutions

#basic join on raw title field
joined_basic = iclr_cleaned.join(
    openalex_df.select(
        F.col("title"),
        F.col("cited_by_count").cast("double")
    ),
    iclr_cleaned.title == openalex_df.title,
    how="inner"
)

#counting by year
print("--- Q5 BASIC RESULT: Matched papers by year ---")
joined_basic.groupBy("year").count().orderBy("year").show()
print(f"Total ICLR papers with OpenAlex data: {joined_basic.count()}")

--- Q5 BASIC RESULT: Matched papers by year ---
+----+-----+
|year|count|
+----+-----+
|2017|  154|
|2018|  309|
|2019|  199|
|2020|  433|
|2021|  423|
|2022|   52|
|2023|   19|
|2024|    1|
|2025|    2|
|2026|    1|
+----+-----+

Total ICLR papers with OpenAlex data: 1593


In [0]:
#Q5: ICLR papers with OpenAlex data by year
#joining by normalised lowercased and trimmed title
#only using papers present in both datasets - 70.9% match rate
#deduplication applied since some ICLR papers matched multiple in openAlex
#max cited_by_count kept per paper

from pyspark.sql.functions import max as spark_max

#normalising titles in both datasets
iclr_norm = iclr_cleaned.withColumn(
    "title_norm", F.lower(F.trim(F.col("title")))
)
openalex_clean = openalex_df.select(
    F.lower(F.trim(F.col("title"))).alias("title_norm"),
    F.col("cited_by_count").cast("double")
)

#deduplicating openAlex before joining
#some papers appear multiple times so keeping highest citation count
openalex_dedup = openalex_clean \
    .groupBy("title_norm") \
    .agg(spark_max("cited_by_count").alias("cited_by_count"))

#inner-joining by normalised title using deduplicated openAlex
joined_dedup = iclr_norm.join(openalex_dedup, on="title_norm", how="inner")

#verifying no duplicates remain
print(f"Total rows: {joined_dedup.count()}")
print(f"Distinct IDs: {joined_dedup.select('id').distinct().count()}")

#counting by year
#answering the question
print("--- Q5 RESULT: Matched papers by year ---")
joined_dedup.groupBy("year").count().orderBy("year").show()
print(f"Total ICLR papers with OpenAlex data: {joined_dedup.count()}")

Total rows: 1602
Distinct IDs: 1602
--- Q5 RESULT: Matched papers by year ---
+----+-----+
|year|count|
+----+-----+
|2017|  153|
|2018|  291|
|2019|  215|
|2020|  439|
|2021|  420|
|2022|   60|
|2023|   19|
|2024|    1|
|2025|    3|
|2026|    1|
+----+-----+

Total ICLR papers with OpenAlex data: 1602


In [0]:
#Q5 EXTENSION: citation analysis of matched papers
from pyspark.sql.functions import stddev, avg, count as spark_count

#joining mean scores and decisions with citation data
#using aliases to avoid ambiguous column names
citation_analysis = mean_scores.alias("ms").join(
    joined_dedup.select("id", "cited_by_count").alias("jd"),
    on="id", how="inner"
).join(
    iclr_cleaned.select("id", "standard_decision",
        F.col("year").alias("paper_year")).alias("ic"),
    on="id", how="inner"
).dropna(subset=["mean_score", "cited_by_count"])

#averaging citations by decision category
print("--- Average citations by decision ---")
citation_analysis.filter(
    F.col("standard_decision").isin(["Accepted", "Rejected"])
).groupBy("standard_decision").agg(
    F.round(avg("cited_by_count"), 1).alias("avg_citations"),
    F.round(F.expr("percentile_approx(cited_by_count, 0.5)"),
        1).alias("median_citations"),
    spark_count("id").alias("num_papers")
).orderBy("standard_decision").show()

#finding correlation between mean score and citation count
corr = citation_analysis.corr("mean_score", "cited_by_count")
print(f"Correlation between mean score and citations: {corr:.4f}")

#averaging citations by year using paper_year to avoid ambiguity
print("--- Average citations by year (matched papers only) ---")
citation_analysis.groupBy("paper_year").agg(
    F.round(avg("cited_by_count"), 1).alias("avg_citations"),
    F.round(F.expr("percentile_approx(cited_by_count, 0.5)"),
        1).alias("median_citations"),
    spark_count("id").alias("num_papers")
).orderBy("paper_year").show()

--- Average citations by decision ---
+-----------------+-------------+----------------+----------+
|standard_decision|avg_citations|median_citations|num_papers|
+-----------------+-------------+----------------+----------+
|         Accepted|         78.2|            22.0|      1315|
|         Rejected|        113.0|            25.0|       241|
+-----------------+-------------+----------------+----------+

Correlation between mean score and citations: 0.0453
--- Average citations by year (matched papers only) ---
+----------+-------------+----------------+----------+
|paper_year|avg_citations|median_citations|num_papers|
+----------+-------------+----------------+----------+
|      2017|        292.4|           108.0|       153|
|      2018|        144.9|            51.0|       287|
|      2019|         59.3|            30.0|       215|
|      2020|         45.2|            22.0|       431|
|      2021|         28.9|            10.0|       420|
|      2022|         21.5|             7

In [0]:
openalex_df.printSchema()
openalex_df.select("citation_normalized_percentile").show(5, truncate=False)

root
 |-- apc_list: struct (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- value: long (nullable = true)
 |    |-- value_usd: long (nullable = true)
 |-- apc_paid: struct (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- value: long (nullable = true)
 |    |-- value_usd: long (nullable = true)
 |-- authorships: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- affiliations: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- institution_ids: array (nullable = true)
 |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |-- raw_affiliation_string: string (nullable = true)
 |    |    |-- author: struct (nullable = true)
 |    |    |    |-- display_name: string (nullable = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- orcid: string (nullable = true)
 |    |    |-- author_position: string (

In [0]:
#Q6: Low-scored papers in the 95th citation percentile
#using citation_normalized_percentile.value from OpenAlex
#mean score < 5 defines a low score paper, consistent with Q2
#citation_normalized_percentile.value >= 0.95 used as pre-computed percentile indicator

#computing mean score per paper and filtering to low-score papers
#reusing mean_scores DataFrame from Q3 rather than recomputing
iclr_q6 = mean_scores.filter(F.col("mean_score") < 5)
print(f"Papers with mean score < 5: {iclr_q6.count()}")

#preparing OpenAlex: extracting title, citation count and normalised percentile value
#deduplicating by keeping the record with the highest percentile value per title
#this ensures papers with duplicate entries retain the valid percentile record
w_oa = Window.partitionBy("title")
openalex_q6 = openalex_df \
    .select(
        F.col("title"),
        F.col("cited_by_count").cast("double"),
        F.col("citation_normalized_percentile.value").alias("percentile_value")
    ) \
    .withColumn("max_percentile", F.max("percentile_value").over(w_oa)) \
    .filter(F.col("percentile_value") == F.col("max_percentile")) \
    .drop("max_percentile") \
    .dropDuplicates(["title"]) \
    .withColumnRenamed("title", "oa_title")

print(f"OpenAlex records after deduplication: {openalex_q6.count()}")

#joining low score ICLR papers to OpenAlex on exact title match
#using iclr_cleaned title field joined to OpenAlex title
q6_joined = iclr_q6.join(
    iclr_cleaned.select("id", "title", "year"),
    on="id",
    how="inner"
).join(
    openalex_q6,
    iclr_cleaned["title"] == openalex_q6["oa_title"],
    how="inner"
).select(
    iclr_q6["id"],
    F.col("title"),
    iclr_q6["year"],
    F.col("mean_score"),
    F.col("cited_by_count"),
    F.col("percentile_value")
)

print(f"Low score papers with OpenAlex citation data: {q6_joined.count()}")

#filtering to papers in the 95th percentile using citation_normalized_percentile.value
q6_result = q6_joined \
    .filter(F.col("percentile_value") >= 0.95) \
    .select("id", "title", "year", "mean_score", "cited_by_count") \
    .dropDuplicates(["id"]) \
    .orderBy(F.col("cited_by_count").desc())

print("\n--- Q6 RESULT: Low-scored but highly cited papers ---")
q6_result.show(truncate=False)
print(f"Total papers identified: {q6_result.count()}")

Papers with mean score < 5: 15045
OpenAlex records after deduplication: 1219
Low score papers with OpenAlex citation data: 62

--- Q6 RESULT: Low-scored but highly cited papers ---
+----------+----------------------------------------------------------------------------------------+----+------------------+--------------+
|id        |title                                                                                   |year|mean_score        |cited_by_count|
+----------+----------------------------------------------------------------------------------------+----+------------------+--------------+
|S19eAF9ee |Structured Sequence Modeling with Graph Convolutional Recurrent Networks                |2017|4.0               |763.0         |
|HkmaTz-0W |Visualizing the Loss Landscape of Neural Nets                                           |2018|4.666666666666667 |514.0         |
|H1bM1fZCW |GradNorm: Gradient Normalization for Adaptive Loss Balancing in Deep Multitask Networks |2018|4.666666

In [0]:
#Q6: extra pre-processing version with deduplication
#ensuring each ICLR paper appears only once with its highest citation count

#joining mean scores with deduplicated citation data
iclr_with_citations = mean_scores.join(
    joined_dedup.select("id", "title_norm", "cited_by_count"),
    on="id",
    how="inner"
).dropna(subset=["mean_score", "cited_by_count"])

print(f"Papers with both score and citation data: {iclr_with_citations.count()}")

#computing 95th percentile on matched papers only
p95 = iclr_with_citations.approxQuantile(
    "cited_by_count", [0.95], 0.01
)[0]
print(f"95th percentile citation threshold: {p95}")

#filtering mean score < 5 AND citations >= 95th percentile
q6_result = iclr_with_citations.filter(
    (F.col("mean_score") < 5) &
    (F.col("cited_by_count") >= p95)
).select("id", "title_norm", "year", "mean_score", "cited_by_count") \
 .dropDuplicates(["id"]) \
 .orderBy(F.col("cited_by_count").desc())

print("\n--- Q6 RESULT: Low-scored but highly cited papers ---")
q6_result.show(truncate=False)
print(f"Total papers found: {q6_result.count()}")

Papers with both score and citation data: 1589
95th percentile citation threshold: 284.0

--- Q6 RESULT: Low-scored but highly cited papers ---
+----------+---------------------------------------------------------------------------------------+----+-----------------+--------------+
|id        |title_norm                                                                             |year|mean_score       |cited_by_count|
+----------+---------------------------------------------------------------------------------------+----+-----------------+--------------+
|HJy_5Mcll |enet: a deep neural network architecture for real-time semantic segmentation           |2017|4.0              |1034.0        |
|Bygq-H9eg |an analysis of deep neural network models for practical applications                   |2017|4.333333333333333|981.0         |
|S19eAF9ee |structured sequence modeling with graph convolutional recurrent networks               |2017|4.0              |763.0         |
|Bk0MRI5lg |bridging n

In [0]:
#Q6 EXTENSION: linear regression predicting citations
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import stddev, avg, count as spark_count

#building dataset using alias year in iclr_cleaned to avoid ambiguity
lr_data = score_stats.join(
    joined_dedup.select("id", "cited_by_count"),
    on="id", how="inner"
).join(
    iclr_cleaned.select(
        "id",
        F.col("year").alias("paper_year")
    ),
    on="id", how="inner"
).withColumn("cited_by_count",
    F.col("cited_by_count").cast("double")
).withColumn("year_norm",
    (F.col("paper_year") - 2017).cast("double")
).dropna(subset=["mean_score", "score_std",
                  "num_reviewers", "cited_by_count"])

print(f"Linear regression dataset: {lr_data.count()} papers")

#assembling features together into one vector
assembler = VectorAssembler(
    inputCols=["mean_score", "score_std",
               "num_reviewers", "year_norm"],
    outputCol="features"
)
ml_data = assembler.transform(lr_data) \
    .select("features", "cited_by_count").dropna()

#splitting training and testing data (80/20)
train, test = ml_data.randomSplit([0.8, 0.2], seed=42)

#fitting linear regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="cited_by_count",
    maxIter=100
)
lr_model = lr.fit(train)

#evaluating the model using testing set
evaluator = RegressionEvaluator(
    labelCol="cited_by_count",
    metricName="r2"
)
r2 = evaluator.evaluate(lr_model.transform(test))

rmse_eval = RegressionEvaluator(
    labelCol="cited_by_count",
    metricName="rmse"
)
rmse = rmse_eval.evaluate(lr_model.transform(test))

print(f"\nR² score: {r2:.4f}")
print(f"RMSE: {rmse:.2f}")
print("\nCoefficients:")
print(f"  mean_score:    {lr_model.coefficients[0]:.4f}")
print(f"  score_std:     {lr_model.coefficients[1]:.4f}")
print(f"  num_reviewers: {lr_model.coefficients[2]:.4f}")
print(f"  year (norm):   {lr_model.coefficients[3]:.4f}")
print(f"  intercept:     {lr_model.intercept:.4f}")

Linear regression dataset: 1589 papers

R² score: 0.0545
RMSE: 169.50

Coefficients:
  mean_score:    8.3797
  score_std:     5.5615
  num_reviewers: 40.9284
  year (norm):   -57.9669
  intercept:     45.6293


In [0]:
#sense checking
print("Q1:", 11349, 17290, 7594)
print("Q2:", 1596)
print("Q3: top paper u1cQYxRI1H score 10.0")
print("Q4: 4 keywords found")
print("Q5:", 1602, "matched papers")
print("Q6:", 8, "papers found")
print("All questions complete with extensions")

Q1: 11349 17290 7594
Q2: 1596
Q3: top paper u1cQYxRI1H score 10.0
Q4: 4 keywords found
Q5: 1602 matched papers
Q6: 8 papers found
All questions complete with extensions


In [0]:
#sanity check

#Q1: all categories should sum to total rows
q1_total = 11349 + 17290 + 7594 + 19673
print(f"Q1 total: {q1_total} (should be 55906): {'✓' if q1_total == 55906 else '✗'}")

#Q2: accepted with low score should be less than total accepted
print(f"Q2: 1596 < 11349 accepted: {'✓' if 1596 < 11349 else '✗'}")
print(f"Q2: 1596 / 11349 = {round(1596/11349*100,1)}% of accepted papers")

#Q3: mean score of 10.0 is maximum possible
print(f"Q3: top score 10.0 <= 10 (max possible): {'✓' if 10.0 <= 10 else '✗'}")

#Q4: output keywords should have count_low < 5 and count_high > 50
print("\nQ4 verification:")
growth.show(truncate=False)

#Q5: matched papers should be less than both dataset sizes
print(f"Q5: 1602 < 55906 ICLR rows: {'✓' if 1602 < 55906 else '✗'}")
print(f"Q5: 1602 < 2245 OpenAlex rows: {'✓' if 1602 < 2245 else '✗'}")

#Q6: all papers should have mean_score < 5 and cited_by_count >= 284
print("\nQ6 verification — all mean_score < 5 and cited_by_count >= 284:")
q6_result.select("mean_score", "cited_by_count").show()

#Q6: checking no paper has mean_score >= 5
invalid_q6 = q6_result.filter(F.col("mean_score") >= 5).count()
print(f"Q6 papers with mean_score >= 5 (should be 0): {invalid_q6}")

#Q6: checking no paper has cited_by_count < 284
below_threshold = q6_result.filter(F.col("cited_by_count") < 284).count()
print(f"Q6 papers below citation threshold (should be 0): {below_threshold}")

Q1 total: 55906 (should be 55906): ✓
Q2: 1596 < 11349 accepted: ✓
Q2: 1596 / 11349 = 14.1% of accepted papers
Q3: top score 10.0 <= 10 (max possible): ✓

Q4 verification:
+----------------------+--------+---------+---------+----------+
|keyword               |year_low|count_low|year_high|count_high|
+----------------------+--------+---------+---------+----------+
|test-time scaling     |2025    |3        |2026     |87        |
|llm                   |2023    |4        |2024     |76        |
|large reasoning models|2025    |1        |2026     |57        |
|spatial reasoning     |2025    |3        |2026     |57        |
+----------------------+--------+---------+---------+----------+

Q5: 1602 < 55906 ICLR rows: ✓
Q5: 1602 < 2245 OpenAlex rows: ✓

Q6 verification — all mean_score < 5 and cited_by_count >= 284:
+-----------------+--------------+
|       mean_score|cited_by_count|
+-----------------+--------------+
|              4.0|        1034.0|
|4.333333333333333|         981.0|
|    

In [0]:
#checking if Other = 0
iclr_cleaned.filter(F.col("standard_decision") == "Other").count()

0

In [0]:
#picking the top paper and verifying its mean score manually
top_paper_id = "u1cQYxRI1H"
iclr_df.filter(F.col("id") == top_paper_id) \
    .select("id", "title", "scores", "year", "decision") \
    .show(truncate=False)


+----------+------------------------------------------------------------------------------------------------------------------------------+----------------+----+-------------+
|id        |title                                                                                                                         |scores          |year|decision     |
+----------+------------------------------------------------------------------------------------------------------------------------------+----------------+----+-------------+
|u1cQYxRI1H|Scaling In-the-Wild Training for Diffusion-based Illumination Harmonization and Editing by Imposing Consistent Light Transport|[10, 10, 10, 10]|2025|Accept (Oral)|
+----------+------------------------------------------------------------------------------------------------------------------------------+----------------+----+-------------+



In [0]:
#manually verifying one keyword from Q4
#"test-time scaling" should have count 3 in 2025 and 87 in 2026
keyword_counts.filter(
    F.col("keyword") == "test-time scaling"
).orderBy("year").show()

+----+-----------------+-----+
|year|          keyword|count|
+----+-----------------+-----+
|2025|test-time scaling|    3|
|2026|test-time scaling|   87|
+----+-----------------+-----+



In [0]:
#counting distinct matched IDs. This should equal total joined count
#if duplicates are above 0 it means some ICLR papers matched multiple openAlex records
distinct_matched = joined_dedup.select("id").distinct().count()
total_joined = joined_dedup.count()
print(f"Total joined rows: {total_joined}")
print(f"Distinct IDs: {distinct_matched}")
print(f"Duplicates: {total_joined - distinct_matched}")


Total joined rows: 1602
Distinct IDs: 1602
Duplicates: 0
